# Act 1 — Why cloud at all?

You want to run software for other people to use. You have two options. Buy servers, rack them in a building you cool and secure, and hire people to look after them — or rent that capacity by the hour from a company that already does all of that. Cloud is the second option, treated as a default.

Everything that follows comes from that one choice — what you give up, what you get, and how the trade plays out at scale.

## What cloud actually means

Cloud computing is the **on-demand delivery of compute, storage, networking, and dozens of other services over the internet**, paid for by what you actually consume. The provider owns the hardware, the buildings, the cooling, the security, the network. You request resources through an API; they appear in seconds; you pay only while they exist.

That single shift — from **owning** capacity to **renting** it — changes almost every architectural decision downstream.

## Traditional IT vs. Cloud

| | Traditional IT | Cloud |
|---|---|---|
| **Hardware** | Buy and own | Rent on-demand |
| **Capacity** | Over-provision for peak | Scale up/down in minutes |
| **Speed** | Weeks to provision | Seconds to provision |
| **Cost model** | Large upfront CapEx | Pay-as-you-go OpEx |
| **Maintenance** | Your team manages racks, power, cooling | Provider handles undifferentiated heavy lifting |
| **Global reach** | Slow, expensive | Deploy worldwide in minutes |

**Old pain points cloud removes:** paying for idle capacity, multi-week procurement, guessing capacity 18 months out, and running a physical data center.

## Three Service Models

Which layer you consume determines how much of the operational stack you still own.

| Model | Provider manages | You manage | AWS examples |
|---|---|---|---|
| **IaaS** — Infrastructure as a Service | Virtualization, networking, hardware | OS, runtime, app, data | EC2, EBS, VPC |
| **PaaS** — Platform as a Service | + OS, + runtime | App and data only | Elastic Beanstalk, RDS |
| **SaaS** — Software as a Service | The entire stack | Just your data / use the app | WorkMail, Chime, QuickSight |

Each step up the stack trades **flexibility** for **less operational burden**.

## Deployment Models

| Model | What it is | When you see it |
|---|---|---|
| **Public** | Provider-owned, multi-tenant | Default for everything in this course |
| **Private** | Single-organization infrastructure | Banks, government, regulated workloads |
| **Hybrid** | Public + private connected over a dedicated link | Most large enterprises |
| **Multi-cloud** | More than one public provider | Vendor-lock-in avoidance, niche service usage |

# Act 2 — Why does AWS have so many "places"?

Look at an AWS map and you see regions, availability zones, edge locations, local zones, wavelength zones, outposts. That is a lot of words for "where your code runs," and on a first read it looks like AWS is collecting jargon.

It isn't. All of it falls out of three pressures the cloud has to handle at once:

- **Latency** — your users are spread around the world and the speed of light is fixed.
- **Sovereignty** — data has legal homes. EU customer data may need to stay in the EU; some workloads must stay inside one country entirely.
- **Failure isolation** — your service cannot go down when one building loses power or one network spine has a bad day.

Each layer of the AWS footprint answers one of those pressures. Read this act as three answers and their specialised variants, not a feature catalogue.

## AWS Global Infrastructure

AWS infrastructure is a set of nested layers. Each layer trades **breadth** for **proximity to the user**.

| Layer | What it is | Count (~2026) | Runs |
|---|---|---|---|
| **Region** | Independent geographic area with its own data centers | 30+ | All AWS services |
| **Availability Zone** | One or more discrete data centers within a region | 3–6 per region | All regional services |
| **Edge Location / PoP** | Small facility for terminating user connections | 600+ | CloudFront, Route 53, WAF, Shield, Lambda@Edge |
| **Local Zone** | Region extension in a metro city | Select cities | EC2, EBS, some RDS / load balancing |
| **Wavelength Zone** | AWS embedded inside a 5G carrier's network | Select carriers | EC2, EBS for ultra-low-latency mobile |
| **Outpost** | AWS rack hardware in *your* data center | Anywhere you ship it | EC2, EBS, S3, RDS, ECS, EKS |

## Regions

A **Region** is a physical geographic area containing a cluster of AWS data centers — Northern Virginia, Ireland, Frankfurt, Mumbai. Each has a code: `us-east-1`, `eu-west-1`, `ap-south-1`. The number is just an identifier; it doesn't count anything.

**Key property: independence.** Regions don't share power, networking, or control planes. A failure in one has no causal connection to another. Most AWS resources live *in* a region — an S3 bucket, an RDS database, a VPC. To have a presence in another region, you deploy there explicitly.

**Global services** are the exception: IAM, Route 53, CloudFront, WAF (for CloudFront), Organizations. The region selector grays out for these.

**`us-east-1` quirk** — new services launch there first, and several global services use it as their home control plane (billing, IAM, CloudFront config). Outages in `us-east-1` occasionally have wider blast radius than other regional outages.

## Availability Zones (AZs)

An **AZ** is one or more discrete data centers, each with its own power, cooling, and physical security.

- **3–6 AZs per region**, named by suffix: `us-east-1a`, `us-east-1b`, `us-east-1c`
- **Tens of miles apart** — far enough that a single fire/flood cannot take more than one
- **Connected by high-bandwidth, low-latency fiber** — network between them is fast and effectively local

**Why they exist:** running across two or three AZs inside a region is the foundation of high availability on AWS. Anywhere you see **multi-AZ** in AWS docs — RDS Multi-AZ, ASG across zones, ALB across zones — this is the principle at work.

## Edge Locations

Edge locations are **not regions** — they don't run EC2 or RDS. They are smaller facilities deployed in many more cities than regions, whose only job is to terminate user connections close to the user.

| Service | What it does at the edge |
|---|---|
| **CloudFront** | Caches and delivers content from the nearest edge |
| **Route 53** | Answers DNS from edge locations |
| **AWS WAF** | Filters HTTP traffic at the edge |
| **AWS Shield** | DDoS protection at the edge |
| **Lambda@Edge** / **CloudFront Functions** | Run code at the edge to customize requests/responses |

## Beyond Regions: Local Zones, Wavelength, Outposts

Three options for when even a region is too far away — each pushes services closer to a different kind of user.

| Option | Direction | Use case |
|---|---|---|
| **Local Zones** | AWS into a *city* | Live media production, interactive gaming, real-time finance |
| **Wavelength** | AWS into a *5G carrier network* | Connected vehicles, smart factories, AR on mobile |
| **Outposts** | AWS into *your data center* | Strict data residency, ultra-low latency to local industrial systems |

# Act 3 — How do you actually talk to AWS?

You know what cloud is and where on the planet AWS runs your code. But how does a request actually leave your laptop and create an EC2 instance, an S3 bucket, a Lambda function?

The answer is mundane and worth understanding once. Every interaction with AWS — every resource, every action — is a signed HTTPS API call to a regional endpoint. The console, the CLI, and the SDKs are three different doors into the same control plane. Which door you pick depends on what you are doing; most engineers use all three in the same day.

## How You Talk to AWS

Every interaction goes through the same control plane — three doors into it.

| Door | What it is | When to use |
|---|---|---|
| **AWS Management Console** | Web UI in a browser | Learning a new service, one-off exploration |
| **AWS CLI** | Commands from your terminal | Scripting, automation glue, ad-hoc ops |
| **SDKs** (e.g. `boto3`) | Programmatic access from code | Application code, infrastructure-as-code, repeatable workflows |

Underneath all three: signed HTTPS requests to a regional API endpoint. The console is a web app calling those endpoints; the CLI is a thin wrapper; the SDK is a typed wrapper.

In [ ]:
# `boto3` knows the full list of AWS regions without making any API call —
# the metadata ships with the library. No credentials needed.
import boto3

regions = boto3.session.Session().get_available_regions("ec2")
print(f"{len(regions)} EC2 regions known to this boto3:")
for r in regions:
    print(" ", r)

In [ ]:
# Once credentials are configured (`aws configure` or env vars),
# you can ask EC2 for the actual AZs in a region.
#
# ec2 = boto3.client("ec2", region_name="us-east-1")
# zones = ec2.describe_availability_zones()["AvailabilityZones"]
# for z in zones:
#     print(z["ZoneName"], "-", z["State"])

# Act 4 — Who owns what, and how do you do it well?

You have now seen what cloud trades, where AWS runs your code, and how you call it. Two questions remain before you can design anything sensibly.

First — when something goes wrong, whose problem is it? AWS owns part of the stack, you own the rest, and the line between them moves depending on which service you are using. That is the **shared responsibility model**.

Second — given the parts that are your problem, what does "good" look like? AWS publishes its answer as the **Well-Architected Framework** — six pillars that act as a vocabulary for the trade-offs you will keep facing.

The act closes with the most concrete first decision either model will force you to make: which region to deploy in.

## Shared Responsibility — Primer

| | AWS is responsible for | You are responsible for |
|---|---|---|
| **Slogan** | Security **of** the cloud | Security **in** the cloud |
| **Always** | Physical buildings, hypervisors, networking gear, host OS, the services they operate | Your data, your IAM users/roles/policies, your firewall rules, your encryption choices |
| **EC2** | Hardware, hypervisor | Guest OS patching, application, network rules |
| **RDS** | Hardware, OS, database engine patching | Users, queries, backups configuration |
| **S3** | Hardware, OS, storage durability | Data, bucket policies, access controls |

**The line moves up the stack as the service gets more managed — but responsibility for *what you put in* and *who you let see it* never goes away.** Notebook 12 covers this in depth.

## Well-Architected Framework — Six Pillars

The lens AWS publishes for reviewing any workload. Not a checklist — a vocabulary for trade-offs.

| Pillar | Goal | Concrete techniques |
|---|---|---|
| **Operational Excellence** | Run and improve systems to deliver value | IaC, observability, runbooks, post-mortems |
| **Security** | Protect data, systems, assets | IAM, encryption, detective controls, incident response |
| **Reliability** | Perform correctly; recover from failure | Multi-AZ, Auto Scaling, DR planning |
| **Performance Efficiency** | Right resources for the job, kept right as demand shifts | Right-sizing, the right database for the workload |
| **Cost Optimization** | Deliver value at the lowest price point | Right-sizing, Reserved/Spot/Savings Plans, kill idle resources |
| **Sustainability** | Minimize environmental impact | Efficient regions, right-sizing, managed services |

Notebook 14 returns to this with concrete service mappings.

## Choosing a Region

Four factors, in roughly this order of weight:

1. **Compliance & data residency** — some regulations require data to stay in a country or economic zone. Often non-negotiable.
2. **Latency** — closest region to your users wins. If users are spread across continents, design multi-region with Route 53 / CloudFront in front.
3. **Service availability** — new services launch in `us-east-1` and a handful of others first, then spread over months/years. Verify every service you need is available in your target.
4. **Pricing** — varies modestly between regions. Tiebreaker, not driver.